# 1. Gathering Data

In [2]:
# Import Libraries
import pandas as pd # For data manipulation
import numpy as np # For numerical operations
import re # For regular expressions
import os # For file handling
import glob # For file handling

In [3]:
# Define directory and target campuses
base_dir = 'Kawan Kampus'

kampus_list = [
    'Institut Pertanian Bogor',
    'STMIK IKMI CIREBON',
    'Universitas Indonesia',
    'UNIVERSITAS MULTI DATA PALEMBANG',
    'Universitas Pendidikan Indonesia Bandung'
]

In [4]:
# Data Cleaning Function
def clean_kawan_kampus_data(df, file_name, nama_kampus):
    # A. Rename columns based on mapping dictionary
    col_map = {
        'qBF1Pd': 'Nama_Tempat',
        'MW4etd': 'Rating',
        'UY7F9': 'Total_Reviews',
        'W4Efsd': 'Kategori_Google',
        'hfpxzc href': 'Google_Maps_Link',
        'ah5Ghc': 'Info_Rute'
    }
    df_clean = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})
    
    # B. Filter required columns for Kawan Kampus project
    cols_to_keep = ['Nama_Tempat', 'Rating', 'Total_Reviews', 'Kategori_Google', 'Google_Maps_Link', 'Info_Rute']
    existing_cols = [col for col in cols_to_keep if col in df_clean.columns]
    df_clean = df_clean[existing_cols].copy()
    
    # C. Drop rows with missing 'Nama_Tempat' values
    if 'Nama_Tempat' in df_clean.columns:
        df_clean = df_clean.dropna(subset=['Nama_Tempat'])
    
    # D. Extract Latitude and Longitude from Google Maps URL
    def extract_lat_long(url):
        if pd.isna(url): return pd.Series([None, None])
        lat_match = re.search(r'!3d(-?\d+\.\d+)', str(url))
        long_match = re.search(r'!4d(-?\d+\.\d+)', str(url))
        return pd.Series([lat_match.group(1) if lat_match else None, 
                          long_match.group(1) if long_match else None])
    
    if 'Google_Maps_Link' in df_clean.columns:
        df_clean[['Latitude', 'Longitude']] = df_clean['Google_Maps_Link'].apply(extract_lat_long)
        
    # E. Format numeric columns for processing
    if 'Total_Reviews' in df_clean.columns:
        df_clean['Total_Reviews'] = df_clean['Total_Reviews'].astype(str).str.replace(r'\D', '', regex=True)
        df_clean['Total_Reviews'] = pd.to_numeric(df_clean['Total_Reviews'], errors='coerce')
        
    if 'Rating' in df_clean.columns:
        df_clean['Rating'] = df_clean['Rating'].astype(str).str.replace(',', '.')
        df_clean['Rating'] = pd.to_numeric(df_clean['Rating'], errors='coerce')
        
    # F. Feature Engineering: Create 'Kategori_Awal' and 'Kampus_Terdekat'
    kategori_mentah = file_name.split('_')[0]
    df_clean['Kategori_Awal'] = kategori_mentah.replace('-', ' ').title()
    df_clean['Kampus_Terdekat'] = nama_kampus
    
    return df_clean

In [5]:
# --- Execute Data Gathering & Initial Cleaning ---
print("Starting Kawan Kampus data processing...")
all_dataframes = []       
total_file_terbaca = 0    

for kampus in kampus_list:
    folder_path = os.path.join(base_dir, kampus)
    
    if not os.path.exists(folder_path):
        print(f"Warning: Directory '{kampus}' not found. Skipping.")
        continue
        
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    
    for file in csv_files:
        nama_file = os.path.basename(file) 
        try:
            df = pd.read_csv(file)         
            df_bersih = clean_kawan_kampus_data(df, nama_file, kampus) 
            all_dataframes.append(df_bersih) 
            total_file_terbaca += 1          
        except Exception as e:
            print(f"Error reading {nama_file}: {e}")

Starting Kawan Kampus data processing...


In [6]:
# --- Merge and Export Master Dataset ---
if len(all_dataframes) > 0:
    df_master = pd.concat(all_dataframes, ignore_index=True)
    baris_awal = len(df_master)
    
    # Remove duplicate rows based on identical locations
    df_master = df_master.drop_duplicates(subset=['Nama_Tempat', 'Latitude', 'Longitude'])
    baris_akhir = len(df_master)
    
    output_name = 'kawankampus_master_dataset.csv'
    df_master.to_csv(output_name, index=False)
    
    print("\n=== Final Data Gathering Report ===")
    print(f"Initial row count : {baris_awal}")
    print(f"Final row count   : {baris_akhir}")
    print(f"Output saved to   : {output_name}")
else:
    print("Error: No data available for processing.")


=== Final Data Gathering Report ===
Initial row count : 4232
Final row count   : 3765
Output saved to   : kawankampus_master_dataset.csv


# 2. Assessing Data

In [7]:
# --- Data Assessment ---
file_path = 'kawankampus_master_dataset.csv'
print(f"Opening file {file_path} for assessment...\n")

try:
    df_audit = pd.read_csv(file_path)
    
    # A. Remove unnecessary columns
    if 'Info_Rute' in df_audit.columns:
        df_audit = df_audit.drop(columns=['Info_Rute'])
        
    # B. Rename columns for consistency
    if 'Kampus_Terdekat' in df_audit.columns:
        df_audit = df_audit.rename(columns={'Kampus_Terdekat': 'Kampus'})
        
    # C. Reorder columns to standard format
    urutan_ideal = [
        'Kampus', 'Nama_Tempat', 'Kategori_Awal', 'Kategori_Google', 
        'Rating', 'Total_Reviews', 'Latitude', 'Longitude', 'Google_Maps_Link'
    ]
    urutan_final = [col for col in urutan_ideal if col in df_audit.columns]
    df_audit = df_audit[urutan_final]
    
    df_audit.to_csv(file_path, index=False)

    # --- Generate Assessment Report ---
    print("=== Kawan Kampus Assessment Report ===\n")
    
    # 1. Missing Values Analysis
    print("1. Missing Values:")
    missing_data = df_audit.isnull().sum()
    print(missing_data[missing_data > 0].to_string() if missing_data.sum() > 0 else "No missing values detected.")
    print("-" * 40)
    
    # 2. Data Types Validation
    print("2. Data Types:")
    print(df_audit[['Rating', 'Total_Reviews', 'Latitude', 'Longitude']].dtypes)
    print("-" * 40)
    
    # 3. Duplicate Coordinates Analysis
    print("3. Duplicate Coordinates:")
    dup_coords = df_audit[df_audit.duplicated(subset=['Latitude', 'Longitude'], keep=False)]
    jml_dup = len(dup_coords)
    print(f"Found {jml_dup} locations with overlapping coordinates.")
    print("-" * 40)
    
    # 4. Category Consistency
    print("4. Category Consistency:")
    print(f"Total Initial Categories : {df_audit['Kategori_Awal'].nunique()}")
    print(f"Total Google Categories  : {df_audit['Kategori_Google'].nunique()}")
    print("-" * 40)
    
    # 5. Outliers Analysis
    print("5. Outliers Analysis (Rating):")
    outlier_rating = df_audit[(df_audit['Rating'] > 5.0) | (df_audit['Rating'] < 1.0)]
    print(f"Found {len(outlier_rating)} locations with out-of-bound ratings (<1.0 or >5.0).")
    
    rating_sempurna_palsu = df_audit[(df_audit['Rating'] == 5.0) & (df_audit['Total_Reviews'] < 5)]
    print(f"Found {len(rating_sempurna_palsu)} locations with 5.0 rating but <5 total reviews.")
    print("-" * 40)
    
    # 6. Geographical Bounds
    print("6. Geographical Bounds (Min/Max Coordinates):")
    batas_lokasi = df_audit.groupby('Kampus')[['Latitude', 'Longitude']].agg(['min', 'max'])
    print(batas_lokasi)
    print("=========================================\n")

except FileNotFoundError:
    print(f"Error: File '{file_path}' not found.")

Opening file kawankampus_master_dataset.csv for assessment...

=== Kawan Kampus Assessment Report ===

1. Missing Values:
Kategori_Google      1
Rating             538
Total_Reviews      538
Latitude             1
Longitude            1
----------------------------------------
2. Data Types:
Rating           float64
Total_Reviews    float64
Latitude         float64
Longitude        float64
dtype: object
----------------------------------------
3. Duplicate Coordinates:
Found 48 locations with overlapping coordinates.
----------------------------------------
4. Category Consistency:
Total Initial Categories : 20
Total Google Categories  : 204
----------------------------------------
5. Outliers Analysis (Rating):
Found 0 locations with out-of-bound ratings (<1.0 or >5.0).
Found 412 locations with 5.0 rating but <5 total reviews.
----------------------------------------
6. Geographical Bounds (Min/Max Coordinates):
                                          Latitude             Longitude 

# 3. Cleaning Data

In [8]:
# --- Data Cleaning ---
file_path = 'kawankampus_master_dataset.csv'
print("Starting Data Cleaning process...\n")

try:
    df_clean = pd.read_csv(file_path)
    baris_awal = len(df_clean)
    
    # 1. Handle missing values in Google Category
    df_clean['Kategori_Google'] = df_clean['Kategori_Google'].fillna('Kategori Tidak Diketahui')
    
    # 2. Impute default metrics for transportation facilities
    is_transportasi = (
        df_clean['Kategori_Google'].str.contains('bus|halte|transit|stasiun', case=False, na=False) | 
        df_clean['Kategori_Awal'].str.contains('bus|halte|transportasi', case=False, na=False)
    )
    df_clean.loc[is_transportasi & df_clean['Rating'].isnull(), 'Rating'] = 0.0
    df_clean.loc[is_transportasi & df_clean['Total_Reviews'].isnull(), 'Total_Reviews'] = 0
    
    # 3. Drop remaining rows with missing metrics
    df_clean = df_clean.dropna(subset=['Rating', 'Total_Reviews'])
    
    # 4. Cast metric column to integer
    df_clean['Total_Reviews'] = df_clean['Total_Reviews'].astype(int)
    
    # 5. Sort dataframe by logical hierarchy
    df_clean = df_clean.sort_values(
        by=['Kampus', 'Kategori_Awal', 'Total_Reviews'], 
        ascending=[True, True, False]
    )
    
    # 6. Reset index
    df_clean = df_clean.reset_index(drop=True)
    
    baris_akhir = len(df_clean)
    baris_dihapus = baris_awal - baris_akhir
    
    # 7. Export cleaned dataset
    output_bersih = 'kawankampus_cleaned_data.csv'
    df_clean.to_csv(output_bersih, index=False)
    
    print("=== Data Cleaning Report ===")
    print(f"Initial row count : {baris_awal}")
    print(f"Rows removed      : {baris_dihapus}")
    print(f"Final row count   : {baris_akhir}")
    print(f"Output saved to   : {output_bersih}")

except FileNotFoundError:
    print(f"Error: File '{file_path}' not found.")

Starting Data Cleaning process...

=== Data Cleaning Report ===
Initial row count : 3765
Rows removed      : 219
Final row count   : 3546
Output saved to   : kawankampus_cleaned_data.csv


# 4. Feature Engineering

In [9]:
# --- Feature Engineering ---
print("Starting Feature Engineering process...\n")

df_ai = pd.read_csv('kawankampus_cleaned_data.csv')

# 1. Create 'Tags' feature for NLP / Cosine Similarity model
df_ai['Tags'] = df_ai['Kampus'] + ' ' + df_ai['Kategori_Awal'] + ' ' + df_ai['Kategori_Google']
df_ai['Tags'] = df_ai['Tags'].str.lower().replace(r'[^\w\s]', ' ', regex=True)

# 2. Create 'Kategori_Popularitas' feature (Binning)
df_ai['Kategori_Popularitas'] = pd.cut(
    df_ai['Total_Reviews'], 
    bins=[-1, 50, 300, float('inf')], 
    labels=['Sepi', 'Lumayan Ramai', 'Sangat Populer']
)

# 3. Create 'Skor_Kepercayaan' feature (Bayesian Average)
C = df_ai['Rating'].mean()
m = 20
v = df_ai['Total_Reviews']
R = df_ai['Rating']
df_ai['Skor_Kepercayaan'] = round((v/(v+m) * R) + (m/(v+m) * C), 2)

# 4. Create Geolocation features (Haversine Formula)
def hitung_jarak(row):
    kampus_str = str(row['Kampus']).lower()
    
    # Define central coordinates for each target campus
    if 'binus' in kampus_str:
        lat1, lon1 = -6.201984724855197, 106.78226989961043
    elif 'brawijaya' in kampus_str or 'ub' in kampus_str:
        lat1, lon1 = -7.952444626759491, 112.61374837203446
    elif 'ugm' in kampus_str or 'gadjah mada' in kampus_str:
        lat1, lon1 = -7.770846277351797, 110.37786876071746
    elif 'unair' in kampus_str or 'airlangga' in kampus_str:
        lat1, lon1 = -7.271863115290546, 112.7590909890101
    elif 'itb' in kampus_str or 'ganesha' in kampus_str:
        lat1, lon1 = -6.890153998683431, 107.6106149890101
    else:
        return 0.0 
        
    lat2, lon2 = row['Latitude'], row['Longitude']
    
    # Calculate Haversine distance in kilometers
    R_bumi = 6371.0 
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    return round(R_bumi * c, 2) 

# Apply distance calculation
df_ai['Jarak_KM'] = df_ai.apply(hitung_jarak, axis=1)

# Categorize distance for UI/UX purposes
df_ai['Kategori_Jarak'] = pd.cut(
    df_ai['Jarak_KM'], 
    bins=[-1, 0.5, 2.0, float('inf')], 
    labels=['Jalan Kaki', 'Perlu Motor', 'Agak Jauh']
)

# 5. Export processed dataset for AI modeling
output_ai = 'kawankampus_feature_engineered.csv'
df_ai.to_csv(output_ai, index=False)

print("=== Feature Engineering Report ===")
print("Status: Success")
print(f"Output saved to: {output_ai}")

Starting Feature Engineering process...

=== Feature Engineering Report ===
Status: Success
Output saved to: kawankampus_feature_engineered.csv
